In [0]:
from pyspark.sql import functions as F
from datetime import date, timedelta

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
SOURCE_TABLE = f"{CATALOG}.bronze.ecb_fx_rates_daily"
TARGET_TABLE = f"{CATALOG}.silver.ecb_fx_rates_daily_silver"
WINDOW_COL   = "rate_date"
OVERLAP_DAYS = 7
KEYS = ["rate_date", "currency", "currency_denom", "exchange_rate_type"]

dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])
mode = dbutils.widgets.get("mode")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  rate_date          DATE,
  currency           STRING,
  currency_denom     STRING,
  exchange_rate_type STRING,
  exchange_rate      DOUBLE,
  silver_updated_at  TIMESTAMP
)
USING DELTA
PARTITIONED BY (rate_date)
""")

def merge_into(table, df, keys):
    df.createOrReplaceTempView("stg")
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    spark.sql(f"""
      MERGE INTO {table} t
      USING stg s
      ON {on}
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

# --- ECB publishes only on working days --------------------------------------
def n_working_days_ago(n):
    d = date.today()
    counted = 0
    while counted < n:
        d -= timedelta(days=1)
        if d.weekday() < 5:        # 0=Mon ... 4=Fri
            counted += 1
    return d

# --- how much bronze to read? --------------------------------------------------
last_loaded = spark.table(TARGET_TABLE).agg(F.max(WINDOW_COL)).first()[0]

if mode == "full" or last_loaded is None:
    src = spark.table(SOURCE_TABLE)
    print("[silver:ecb] reading ALL bronze rows (first run or mode=full)")
else:
    src = (spark.table(SOURCE_TABLE)
           .where(F.col(WINDOW_COL) >= F.date_sub(F.lit(last_loaded), OVERLAP_DAYS)))
    print(f"[silver:ecb] incremental: bronze from {last_loaded} (-{OVERLAP_DAYS}d overlap)")

# --- keep only the columns we want -------------------------------------------
out = (src
    .select(
        F.to_date("rate_date").alias("rate_date"),
        F.col("currency"),
        F.col("currency_denom"),
        F.col("exchange_rate_type"),
        F.col("exchange_rate").cast("double"),
        F.current_timestamp().alias("silver_updated_at"),
    )
    .dropDuplicates(KEYS)
    .cache())

# --- data quality: fail loudly BEFORE writing ---------------------------------
n = out.count()
assert n > 0, "[silver:ecb] no rows read from bronze - check bronze table"

for k in KEYS:
    bad = out.where(F.col(k).isNull()).count()
    assert bad == 0, f"[silver:ecb] {bad} rows with null in key {k}"

# some ECB currencies trade at thousands per EUR
bad_rate = out.where(
    F.col("exchange_rate").isNull()
    | (F.col("exchange_rate") < 0.0001)
    | (F.col("exchange_rate") > 1000000)
).count()
assert bad_rate == 0, f"[silver:ecb] {bad_rate} rows of exchange_rate outside [0.0001, 1000000]"

# working days only (ECB never publishes on weekends/holidays)
oldest_ok = n_working_days_ago(7)
fresh = out.where(F.col("rate_date") >= oldest_ok).count()
assert fresh > 0, (f"[silver:ecb] freshness fail: no rate_date on/after {oldest_ok} "
                   f"- ECB daily feed looks dead")

# --- write --------------------------------------------------------------------
merge_into(TARGET_TABLE, out, KEYS)
print(f"[silver:ecb] processed={n}, total={spark.table(TARGET_TABLE).count()}")
out.unpersist()